In [0]:
from pyspark.sql.functions import (
    col, trim, lower, upper, to_timestamp, current_timestamp,
    row_number, lit, expr, max as spark_max
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS metadata")

table to store watermark for incremental load

In [0]:
%sql
CREATE TABLE IF NOT EXISTS metadata.pipeline_watermark (
    table_name STRING,
    last_processed_timestamp TIMESTAMP
)
USING DELTA;

get watermark fucntion

In [0]:
def get_last_watermark(table_name):
    result = spark.sql(f"""
        SELECT last_processed_timestamp
        FROM metadata.pipeline_watermark
        WHERE table_name = '{table_name}'
    """).collect()

    if len(result) == 0 or result[0]["last_processed_timestamp"] is None:
        return "1900-01-01 00:00:00"

    return result[0]["last_processed_timestamp"]

update watermark

In [0]:
def update_watermark(table_name, new_watermark):
    if new_watermark is None:
        return

    watermark_df = spark.createDataFrame(
        [(table_name, new_watermark)],
        ["table_name", "last_processed_timestamp"]
    )

    if not spark.catalog.tableExists("metadata.pipeline_watermark"):
        watermark_df.write.format("delta").mode("overwrite").saveAsTable("metadata.pipeline_watermark")
    else:
        target = DeltaTable.forName(spark, "metadata.pipeline_watermark")

        (
            target.alias("target")
            .merge(
                watermark_df.alias("source"),
                "target.table_name = source.table_name"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

Transformation fucntion for silver layer

In [0]:
def build_clean_df(table_name, cfg):
    last_watermark = get_last_watermark(table_name)

    df = (
        spark.table(cfg["source_table"])
        .filter(col("ingestion_timestamp") > lit(last_watermark))
    )

    if df.limit(1).count() == 0:
        return None, None

    selected_cols = []

    for target_col, rule in cfg["column_map"].items():
        source_col = rule["source"]
        data_type = rule.get("type")
        transform = rule.get("transform")

        column_expr = col(source_col)

        if transform == "trim":
            column_expr = trim(column_expr)
        elif transform == "lower_trim":
            column_expr = lower(trim(column_expr))
        elif transform == "upper_trim":
            column_expr = upper(trim(column_expr))
        elif transform == "timestamp":
            column_expr = to_timestamp(column_expr)

        if data_type:
            column_expr = column_expr.cast(data_type)

        selected_cols.append(column_expr.alias(target_col))

    clean_df = df.select(*selected_cols)

    for condition in cfg["filters"]:
        clean_df = clean_df.filter(condition)

    window_spec = Window.partitionBy(*cfg["dedup_keys"]).orderBy(
        col("ingestion_timestamp").desc(),
        col("source_file").desc()
    )

    dedup_df = (
        clean_df
        .withColumn("rn", row_number().over(window_spec))
        .filter(col("rn") == 1)
        .drop("rn")
        .withColumn("silver_processed_at", current_timestamp())
    )

    new_watermark = df.agg(spark_max("ingestion_timestamp")).collect()[0][0]

    return dedup_df, new_watermark

merge fucntion for incremental load 

In [0]:
def merge_to_silver(source_df, target_table, merge_keys):
    if not spark.catalog.tableExists(target_table):
        (
            source_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(target_table)
        )
    else:
        target = DeltaTable.forName(spark, target_table)

        merge_condition = " AND ".join(
            [f"target.{key} = source.{key}" for key in merge_keys]
        )

        (
            target.alias("target")
            .merge(source_df.alias("source"), merge_condition)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

Table Configuration

In [0]:
silver_configs = {
    "orders": {
        "source_table": "bronze.orders_ext",
        "target_table": "silver.orders",
        "dedup_keys": ["order_id"],
        "filters": [
            col("order_id").isNotNull(),
            col("customer_id").isNotNull()
        ],
        "column_map": {
            "order_id": {"source": "order_id", "transform": "trim", "type": "string"},
            "customer_id": {"source": "customer_id", "transform": "trim", "type": "string"},
            "order_status": {"source": "order_status", "transform": "lower_trim", "type": "string"},
            "order_purchase_timestamp": {"source": "order_purchase_timestamp", "transform": "timestamp"},
            "order_approved_at": {"source": "order_approved_at", "transform": "timestamp"},
            "order_delivered_carrier_date": {"source": "order_delivered_carrier_date", "transform": "timestamp"},
            "order_delivered_customer_date": {"source": "order_delivered_customer_date", "transform": "timestamp"},
            "order_estimated_delivery_date": {"source": "order_estimated_delivery_date", "transform": "timestamp"},
            "source_system": {"source": "source_system"},
            "ingestion_batch_id": {"source": "ingestion_batch_id"},
            "ingestion_timestamp": {"source": "ingestion_timestamp"},
            "source_file": {"source": "source_file"}
        }
    },

    "customers": {
        "source_table": "bronze.customers_ext",
        "target_table": "silver.customers",
        "dedup_keys": ["customer_id"],
        "filters": [
            col("customer_id").isNotNull()
        ],
        "column_map": {
            "customer_id": {"source": "customer_id", "transform": "trim", "type": "string"},
            "customer_unique_id": {"source": "customer_unique_id", "transform": "trim", "type": "string"},

            # Keep ZIP as STRING, not INT.
            # ZIP codes are identifiers, not numbers.
            "customer_zip_code_prefix": {"source": "customer_zip_code_prefix", "transform": "trim", "type": "string"},

            "customer_city": {"source": "customer_city", "transform": "lower_trim", "type": "string"},
            "customer_state": {"source": "customer_state", "transform": "upper_trim", "type": "string"},
            "customer_segment": {"source": "customer_segment", "transform": "trim", "type": "string"},
            "source_system": {"source": "source_system"},
            "ingestion_batch_id": {"source": "ingestion_batch_id"},
            "ingestion_timestamp": {"source": "ingestion_timestamp"},
            "source_file": {"source": "source_file"}
        }
    },

    "order_items": {
        "source_table": "bronze.order_items_ext",
        "target_table": "silver.order_items",
        "dedup_keys": ["order_id", "order_item_id"],
        "filters": [
            col("order_id").isNotNull(),
            col("order_item_id").isNotNull(),
            col("product_id").isNotNull()
        ],
        "column_map": {
            "order_id": {"source": "order_id", "transform": "trim", "type": "string"},
            "order_item_id": {"source": "order_item_id", "type": "int"},
            "product_id": {"source": "product_id", "transform": "trim", "type": "string"},
            "seller_id": {"source": "seller_id", "transform": "trim", "type": "string"},
            "shipping_limit_date": {"source": "shipping_limit_date", "transform": "timestamp"},
            "price": {"source": "price", "type": "double"},
            "freight_value": {"source": "freight_value", "type": "double"},
            "source_system": {"source": "source_system"},
            "ingestion_batch_id": {"source": "ingestion_batch_id"},
            "ingestion_timestamp": {"source": "ingestion_timestamp"},
            "source_file": {"source": "source_file"}
        }
    },

    "products": {
    "source_table": "bronze.products_ext",
    "target_table": "silver.products",
    "dedup_keys": ["product_id"],
    "filters": [
        col("product_id").isNotNull()
    ],
    "column_map": {
        "product_id": {"source": "product_id", "transform": "trim", "type": "string"},
        "product_category_name": {"source": "product_category_name", "transform": "lower_trim", "type": "string"},
        "product_name_length": {"source": "product_name_lenght", "type": "double"},
        "product_description_length": {"source": "product_description_lenght", "type": "double"},
        "product_photos_qty": {"source": "product_photos_qty", "type": "double"},
        "product_weight_g": {"source": "product_weight_g", "type": "double"},
        "product_length_cm": {"source": "product_length_cm", "type": "double"},
        "product_height_cm": {"source": "product_height_cm", "type": "double"},
        "product_width_cm": {"source": "product_width_cm", "type": "double"},
        "product_brand": {"source": "product_brand", "transform": "lower_trim", "type": "string"},
        "source_system": {"source": "source_system"},
        "ingestion_batch_id": {"source": "ingestion_batch_id"},
        "ingestion_timestamp": {"source": "ingestion_timestamp"},
        "source_file": {"source": "source_file"}
    }
  }
}

silver data load for all tables 

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS metadata")

for table_name, cfg in silver_configs.items():
    print(f"Processing silver table: {table_name}")

    clean_df, new_watermark = build_clean_df(table_name, cfg)

    if clean_df is None:
        print(f"No new records found for {table_name}")
        continue

    merge_to_silver(
        source_df=clean_df,
        target_table=cfg["target_table"],
        merge_keys=cfg["dedup_keys"]
    )

    update_watermark(table_name, new_watermark)

    print(f"Completed silver table: {table_name}")